# SingleTaskMultiFidelityGP: Cost-Aware Multi-Fidelity Bayesian Optimization

低コスト・低精度の評価と高コスト・高精度の評価を統合し、target fidelity `s=1.0` の最適化へ接続します。

## このNotebookで確認すること

- fidelityを構造入力として持つsurrogate
- fidelityごとのposterior
- target fidelity上のcurrent terminal value
- target projectionとevaluation fidelityの違い
- fidelity-dependent cost model
- `qMultiFidelityKnowledgeGradient`によるvalue of information
- one-shot acquisition optimizationによる`(x_next, s_next)`生成
- candidate評価とdata update

関連理論:

- [Multi-Fidelity](../../docs/theory/06_multi_fidelity.md)
- [Multi-Fidelity / Cost-aware Acquisition](../../docs/theory/acquisition/11_multifidelity_cost_aware.md)

```text
(x, fidelity) data -> MF surrogate -> posterior
                                -> target projection + cost
                                -> qMFKG -> optimize (x, fidelity)
                                -> evaluation -> data update
```

低fidelity自体の最大化が目的ではありません。最終decisionはtarget fidelityで行います。

## 1. このモデルを使う場面

同じ設計に対して、粗いシミュレーション / 高精度シミュレーション、小規模試験 / 実機試験、簡易測定 / 精密測定など、複数の評価精度・コストが存在する場合に向いています。

fidelity間に相関があることが重要です。


In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.analytic import PosteriorMean
from botorch.acquisition.cost_aware import InverseCostWeightedUtility
from botorch.acquisition.knowledge_gradient import qMultiFidelityKnowledgeGradient
from botorch.fit import fit_gpytorch_mll
from botorch.models.cost import AffineFidelityCostModel
from botorch.optim import optimize_acqf

from robotorchan.models import SingleTaskMultiFidelityGP
from robotorchan.optim import optimize_acqf as optimize_robotorchan_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### Fidelityとは何か

*fidelity* は、同じ設計条件 $x$ をどの程度の精度・計算コストで評価するかを表します。この例では $s=0.25$ を低忠実度、$s=0.60$ を中忠実度、$s=1.0$ を目標（高）忠実度とします。

重要なのは、**低忠実度の値が高い設計を探すことが目的ではない**点です。最終的に最大化したいのは $f(x,1)$ であり、低忠実度の観測はその推定を改善するために使います。

低忠実度が安くても高忠実度と無関係なら役立ちません。両者の相関があり、精度とコストの差が十分にある場合にMulti-Fidelity BOが有効です。

## 2. 合成データの作成

入力の0列目を設計変数 `x`、1列目をfidelity `s` とします。`s=1.0` を目標fidelityとし、低fidelityほど少しバイアスが入る関数を人工的に作ります。


In [ ]:
def objective(x: torch.Tensor, fidelity: torch.Tensor) -> torch.Tensor:
    true_value = torch.sin(2 * torch.pi * x) + 0.25 * torch.cos(4 * torch.pi * x)
    bias = (1.0 - fidelity) * (0.8 * (x - 0.5) + 0.25)
    return true_value + bias


x_low = torch.linspace(0.0, 1.0, 12).unsqueeze(-1)
s_low = torch.full_like(x_low, 0.25)
x_mid = torch.linspace(0.05, 0.95, 8).unsqueeze(-1)
s_mid = torch.full_like(x_mid, 0.60)
x_high = torch.tensor([[0.10], [0.30], [0.55], [0.75], [0.92]])
s_high = torch.ones_like(x_high)

train_X = torch.cat(
    [
        torch.cat([x_low, s_low], dim=-1),
        torch.cat([x_mid, s_mid], dim=-1),
        torch.cat([x_high, s_high], dim=-1),
    ],
    dim=0,
)
train_Y = objective(train_X[:, :1], train_X[:, 1:2])

print(train_X.shape, train_Y.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for fidelity in (0.25, 0.60, 1.00):
    mask = torch.isclose(train_X[:, 1], torch.tensor(fidelity))
    ax.scatter(
        train_X[mask, 0],
        train_Y[mask, 0],
        label=f"observations: fidelity={fidelity:.2f}",
    )

x_plot = torch.linspace(0, 1, 300).unsqueeze(-1)
y_true = objective(x_plot, torch.ones_like(x_plot))
ax.plot(x_plot.squeeze(), y_true.squeeze(), label="target function: fidelity=1.0")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

### なぜ入力にfidelity列を追加するのか

通常のGPは $f(x)$ を予測しますが、Multi-Fidelity GPでは $f(x,s)$ を予測します。`train_X` の1列目が設計変数、2列目がfidelityです。`data_fidelities=[1]` は**Pythonの列番号1をfidelityとして扱う**指定です。

この列は単なる説明変数やカテゴリラベルではありません。異なる忠実度の観測が互いに情報を与える構造を、Multi-Fidelity GPのカーネルで表現します。なお、`s` が連続的に実行可能か、離散的な選択肢だけかは、候補最適化側でも別途指定する必要があります。

## 3. モデル構築

fidelityは入力の1列目なので `data_fidelities=[1]` と指定します。

robotorchanのwrapperはBoTorchの予測挙動を維持しつつ、`raw_*` と `make_mll()` を追加します。


In [ ]:
model = SingleTaskMultiFidelityGP(
    train_X=train_X,
    train_Y=train_Y,
    data_fidelities=[1],
)

model


## 4. robotorchan共通API


In [ ]:
print("raw_train_X shape:", model.raw_train_X.shape)
print("raw_train_Y shape:", model.raw_train_Y.shape)
print("raw_train_Yvar:", model.raw_train_Yvar)
print("raw_data_names:", model.raw_data_names)
print("supports_mll:", model.supports_mll)


## 5. モデル学習

Exact GPの標準 `fit()` を利用します。内部では `make_mll()` とBoTorchの最適化を使用します。


In [ ]:
model.fit()


### Fidelity間で情報を共有する仕組み

同じ設計 $x$ でも、忠実度によって観測値 $f(x,s)$ は異なります。Multi-Fidelity GPは、低忠実度と高忠実度の観測をまとめて学習し、各 $(x,s)$ における予測平均 $\mu(x,s)$ と予測分散 $\sigma^2(x,s)$ を返します。

高忠実度の観測が少ない領域でも、相関のある低忠実度データから予測が改善する可能性があります。ただし、相関が弱い場合や低忠実度に強い系統誤差がある場合は期待通りに改善しないこともあります。

## 6. 複数fidelityでの事後分布予測

同じ `x` に対しfidelityを変えて事後分布を比較します。


In [ ]:
model.eval()
x_test = torch.linspace(0.0, 1.0, 250).unsqueeze(-1)
predictions = {}

with torch.no_grad():
    for fidelity in (0.25, 0.60, 1.00):
        fidelity_column = torch.full_like(x_test, fidelity)
        test_X = torch.cat([x_test, fidelity_column], dim=-1)
        posterior = model.posterior(test_X)
        mean = posterior.mean.squeeze(-1)
        std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)
        predictions[fidelity] = (mean, std)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for fidelity, (mean, _) in predictions.items():
    ax.plot(
        x_test.squeeze(),
        mean,
        label=f"posterior mean: fidelity={fidelity:.2f}",
    )

target = objective(x_test, torch.ones_like(x_test))
ax.plot(x_test.squeeze(), target.squeeze(), linestyle="--", label="true target fidelity")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()


### 目標fidelityでの不確実性

最終的に意思決定するのは $s=1$ なので、特に $\mu(x,1)$ と $\sigma(x,1)$ を確認します。図の帯は $\mu(x,1)\pm1.96\sigma(x,1)$ で表した、**モデル上の潜在関数に関する約95%区間**です。

これは観測ノイズを含む将来の測定値の予測区間とは異なり、真の関数を必ず含む保証でもありません。この合成例で描く破線の真の目標関数は説明用であり、実際のブラックボックス最適化では通常は未知です。

## 7. 目標fidelityでの不確かさ

実際の最適化では、最終的に興味のあるfidelity（ここでは `s=1.0`）での予測が重要です。


In [ ]:
mean, std = predictions[1.00]
lower = mean - 1.96 * std
upper = mean + 1.96 * std

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_test.squeeze(), mean, label="posterior mean at fidelity=1.0")
ax.fill_between(x_test.squeeze(), lower, upper, alpha=0.2, label="95% interval")
target = objective(x_test, torch.ones_like(x_test)).squeeze(-1)
ax.plot(x_test.squeeze(), target, linestyle="--", label="true target")
high_mask = torch.isclose(train_X[:, 1], torch.tensor(1.0))
ax.scatter(
    train_X[high_mask, 0],
    train_Y[high_mask, 0],
    label="high-fidelity observations",
)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

### `current_value` は何を表すか

Knowledge Gradient（KG）は、次の観測によって**最終的な意思決定の価値がどれだけ上がるか**を評価します。その基準になるのが、観測前の目標忠実度上の最良予測値です。

$$V(\mathcal D)=\max_x\mathbb E[f(x,1)\mid\mathcal D]$$

このNotebookの `PosteriorMean(model)` と `fixed_features={1: 1.0}` は、この最大化を目標忠実度に限定するためのものです。`current_value` は観測済みの最大値そのものではなく、**現在のモデルに基づく最良の最終判断の価値**です。

## 8. Target-fidelity current value

MFKGは候補を観測した後のterminal decision改善を評価します。その基準となる`current_value`はtarget fidelity `s=1.0`上で計算します。

In [ ]:
target_fidelity = 1.0
target_mean = PosteriorMean(model)
target_bounds = torch.tensor([[0.0, 1.0], [1.0, 1.0]])
incumbent, current_value = optimize_acqf(
    acq_function=target_mean,
    bounds=target_bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
    fixed_features={1: target_fidelity},
)

assert torch.all(incumbent[..., 1] == target_fidelity)
print("target-fidelity incumbent:", incumbent)
print("current terminal value:", current_value)

### Projectionと実際の評価は異なる

`project(X)` は $(x,s)$ を $(x,1)$ に写し、**最終判断をどの忠実度で行うか**を指定します。`project` を呼んでも実験は実行されず、高忠実度の測定値が得られるわけではありません。

評価コストはここでは $c(s)=0.1+s$ としています。`InverseCostWeightedUtility` は情報の価値とコストを組み合わせ、安価な低忠実度評価を選ぶ余地を作ります。コストの単位・形状は実験に合わせて設定する必要があります。

## 9. Projection and fidelity-dependent cost

`project(X)`は候補を実際にtarget fidelityで評価する操作ではありません。候補`(x, s)`の観測価値を、最終decisionを行う`(x, 1.0)`へ結び付けるための写像です。costはfidelityが高いほど増える簡単なaffine modelを使います。

In [ ]:
def project(X: torch.Tensor) -> torch.Tensor:
    projected = X.clone()
    projected[..., 1] = target_fidelity
    return projected


cost_model = AffineFidelityCostModel(
    fidelity_weights={1: 1.0},
    fixed_cost=0.1,
)
cost_utility = InverseCostWeightedUtility(cost_model=cost_model)

example = torch.tensor([[[0.4, 0.35]]])
print("candidate:", example)
print("projected terminal point:", project(example))

### Knowledge GradientとEIの違い

Notebook 01・02で使ったExpected Improvement（EI）は、次の観測による**直接的な改善**を評価します。一方、Knowledge Gradient（KG）は、候補 $(x,s)$ を観測してモデルを更新したと仮定したときの、**目標忠実度での将来の最良判断の改善**を評価します。

$$\operatorname{KG}(x,s)\approx\mathbb E\bigl[V(\mathcal D\cup\{(x,s,Y)\})\mid\mathcal D\bigr]-V(\mathcal D)$$

`qMultiFidelityKnowledgeGradient` は仮想的な観測（fantasy）を使ってこの期待値を近似します。`num_fantasies=4` はNotebookを軽く実行するための設定であり、精度重視の実務では計算時間と推定の安定性を比較して調整します。

## 10. Multi-Fidelity Knowledge Gradient

`qMultiFidelityKnowledgeGradient`はfantasy modelを使うone-shot acquisitionです。ここではruntimeを抑えるため`num_fantasies=4`とします。

In [ ]:
mf_acqf = qMultiFidelityKnowledgeGradient(
    model=model,
    num_fantasies=4,
    current_value=current_value,
    cost_aware_utility=cost_utility,
    project=project,
)

print("public q:", 1)
print("augmented q:", mf_acqf.get_augmented_q_batch_size(q=1))

### One-shot獲得関数を最適化するときの注意

MFKGは内部で仮想的な観測後の判断を同時に最適化する **one-shot acquisition** です。そのため内部の最適化変数は、外部へ返す実験候補数 `q=1` より多くなります。`get_augmented_q_batch_size(q=1)` はこの内部サイズを確認するためのAPIです。

`optimize_robotorchan_acqf` はこの獲得関数を最適化し、実際に評価する候補 `(x_next, s_next)` を返します。候補の忠実度を目標の `1.0` に固定していないことに注目してください。

## 11. Optimize design and fidelity together

candidate searchでは`x`とfidelity `s`の両方を最適化します。target projectionがあるからといってcandidate fidelityを`1.0`へ固定するわけではありません。

In [ ]:
mf_bounds = torch.tensor([[0.0, 0.25], [1.0, 1.0]])
candidate, acq_value = optimize_robotorchan_acqf(
    acq_function=mf_acqf,
    bounds=mf_bounds,
    q=1,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=32,
)

assert candidate.shape == (1, 2)
assert 0.0 <= candidate[0, 0].item() <= 1.0
assert 0.25 <= candidate[0, 1].item() <= 1.0
print("candidate (x, fidelity):", candidate)
print("acquisition value:", acq_value)

## 12. Candidate evaluation and data update

candidateは選ばれたfidelityで実際に評価します。projection後のtarget fidelity値を観測したことにはしません。

In [ ]:
candidate_Y = objective(candidate[:, :1], candidate[:, 1:2])
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

assert candidate_Y.shape == (1, 1)
assert updated_X.shape == (train_X.shape[0] + 1, 2)
assert updated_Y.shape == (train_Y.shape[0] + 1, 1)
print("observed candidate:", candidate)
print("observed value:", candidate_Y)
print("updated shapes:", updated_X.shape, updated_Y.shape)

## 13. Decision semantics

```text
candidate (x, s)
    │
    ├─ evaluation cost c(s)
    ├─ actual observation at s
    │
    └─ fantasy update
           ↓
      project to target s=1
           ↓
      terminal decision value
```

`SingleTaskMultiFidelityGP`だけではMFBO policyは決まりません。target fidelity、projection、cost、acquisition、candidate optimizationが揃って初めて「どのxをどのfidelityで次に評価するか」を決められます。

## 14. When to use and caveats

- low fidelityがtargetについて情報を持ち、高fidelityより十分安い場合に有効です。
- fidelityが単なる説明変数やnominal categoryなら別モデルを検討します。
- この例はcontinuous fidelityです。実行可能値が`{0.25, 0.60, 1.0}`だけならoptimizer側でもその集合を守る必要があります。
- `qMultiFidelityKnowledgeGradient`はone-shot acquisitionなので、明示的initial conditionsを使う場合はaugmented q dimensionに合わせます。
- cost-aware greedy policyが総budgetに対して常に最適とは限りません。
- mixed categorical + fidelityでは専用のmixed one-shot optimization contractを使います。

## 15. References

詳細な数理、projection、cost-aware utility、continuous/discrete fidelityの違いは冒頭のTheory章を参照してください。

### 追加モデルは何を比較しているのか

ここからは、MAP-SAAS、次元削減（PCA・PLS・ランダム射影）、カテゴリ混合などのMulti-Fidelityサロゲートを使った**API利用例**です。

上のMFKGとは異なり、この節では **target fidelity `s=1` に固定したEI** で候補を選びます。つまり「どのfidelityで評価するか」は最適化せず、モデルごとの学習・予測・目標忠実度での候補提案を確認します。両節の獲得関数値や評価コストを直接比較して、手法の優劣を判断しないでください。

## 16. Additional multi-fidelity surrogates

残りの多忠実度モデルについて、既存の `make_mll()` を使った学習、Posterior、および候補提案と実際の忠実度での評価を確認します。候補は各モデルの目標忠実度 `s=1` に固定し、`qLogExpectedImprovement` を最適化します。この節はcost-aware MFKGではなく、target-fidelity EIの例です。


In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement

from robotorchan.models import (
    AdditiveMapSaasMultiFidelityGP,
    MapSaasMultiFidelityGP,
    MixedSingleTaskMultiFidelityGP,
    PCAMultiFidelityGP,
    PLSMultiFidelityGP,
    RandomProjectionMultiFidelityGP,
)

extra_models = {
    "Additive MAP-SAAS MF": AdditiveMapSaasMultiFidelityGP(
        train_X, train_Y, data_fidelities=[1], num_taus=2
    ),
    "MAP-SAAS MF": MapSaasMultiFidelityGP(
        train_X, train_Y, data_fidelities=[1]
    ),
    "PCA MF": PCAMultiFidelityGP(
        train_X, train_Y, n_components=1, data_fidelities=[1]
    ),
    "PLS MF": PLSMultiFidelityGP(
        train_X, train_Y, n_components=1, data_fidelities=[1]
    ),
    "Random projection MF": RandomProjectionMultiFidelityGP(
        train_X, train_Y, n_components=1, data_fidelities=[1]
    ),
}
mixed_X = torch.cat(
    [train_X[:, :1], torch.arange(train_X.shape[0]).remainder(2)
     .to(train_X).unsqueeze(-1), train_X[:, 1:2]],
    dim=-1,
)
mixed_Y = train_Y + 0.1 * mixed_X[:, 1:2]
extra_models["Mixed MF"] = MixedSingleTaskMultiFidelityGP(
    mixed_X, mixed_Y, cat_dims=[1], data_fidelities=[2]
)
for name, surrogate in extra_models.items():
    if surrogate.supports_fit:
        surrogate.fit()
    else:
        fit_gpytorch_mll(surrogate.make_mll())
    surrogate.eval()
    is_mixed = name == "Mixed MF"
    observed_Y = mixed_Y if is_mixed else train_Y
    fidelity_index = 2 if is_mixed else 1
    acq = qLogExpectedImprovement(model=surrogate, best_f=observed_Y.max())
    candidate_bounds = torch.tensor(
        [[0.0, 0.0, 1.0], [1.0, 1.0, 1.0]]
        if is_mixed else [[0.0, 1.0], [1.0, 1.0]]
    )
    if is_mixed:
        options = []
        for category in (0.0, 1.0):
            point, value = optimize_acqf(
                acq_function=acq, bounds=candidate_bounds,
                q=1, num_restarts=2, raw_samples=16,
                fixed_features={1: category, fidelity_index: 1.0},
            )
            options.append((value.item(), point))
        _, candidate = max(options, key=lambda item: item[0])
    else:
        candidate, _ = optimize_acqf(
            acq_function=acq, bounds=candidate_bounds,
            q=1, num_restarts=2, raw_samples=16,
            fixed_features={fidelity_index: 1.0},
        )
    posterior = surrogate.posterior(candidate)
    candidate_Y = objective(candidate[:, :1], candidate[:, fidelity_index:])
    if is_mixed:
        candidate_Y = candidate_Y + 0.1 * candidate[:, 1:2]
    assert posterior.mean.shape[-1] == 1
    assert candidate[0, fidelity_index].item() == 1.0
    assert torch.isfinite(candidate_Y).all()
    print(name, "candidate:", candidate, "objective:", candidate_Y)
